In [1]:
from pathlib import Path
import runpy
import pandas as pd
import numpy as np
import json
import zipfile
import textwrap


In [2]:
from pathlib import Path

project_dir = Path.cwd().resolve()


def find_project_root(start_dir: Path) -> Path:
    """Locate the project root by checking the current folder and its parents."""
    for candidate in [start_dir, *start_dir.parents]:
        if (candidate / "generate_synthetic_student_journey.py").exists():
            return candidate.resolve()

        nested_candidate = candidate / "student_journey_graph_project"
        if (nested_candidate / "generate_synthetic_student_journey.py").exists():
            return nested_candidate.resolve()

    raise FileNotFoundError(
        "Could not locate the project root containing generate_synthetic_student_journey.py."
    )


project_dir = find_project_root(project_dir)
data_dir = project_dir / "data"
sql_dir = project_dir / "sql"

data_dir.mkdir(parents=True, exist_ok=True)
sql_dir.mkdir(parents=True, exist_ok=True)


In [4]:

# Run the deterministic synthetic-data generator created for this project.
namespace = runpy.run_path(str(project_dir / "generate_synthetic_student_journey.py"))

table_names = [
    "students",
    "programs",
    "courses",
    "terms",
    "institutions",
    "services",
    "enrollments",
    "student_terms",
    "service_usage",
    "program_history",
    "awards",
    "transfers",
    "program_requirements",
    "course_prerequisites",
]

tables = {name: namespace[name] for name in table_names}


In [5]:

# Save every project table as a CSV.
for name, df in tables.items():
    df.to_csv(data_dir / f"{name}.csv", index=False)

# Build a data dictionary from the generated DataFrames.
descriptions = {
    "students": "One fictional row per student. Contains entry characteristics and the simulated final outcome.",
    "programs": "Academic programs used as graph nodes.",
    "courses": "Course catalog used as graph nodes.",
    "terms": "Academic terms used as graph nodes.",
    "institutions": "Fictional transfer destinations used as graph nodes.",
    "services": "Student-support services used as graph nodes.",
    "enrollments": "Student-to-course edges with term, grade, completion, modality, and attempted credits.",
    "student_terms": "Student-to-term edges summarizing term and cumulative progress.",
    "service_usage": "Student-to-support-service edges with visit context.",
    "program_history": "Student-to-program declaration edges, including program changes.",
    "awards": "Student-to-program award edges.",
    "transfers": "Student-to-institution transfer edges.",
    "program_requirements": "Program-to-course requirement edges.",
    "course_prerequisites": "Course-to-course prerequisite or sequence edges.",
}

column_notes = {
    "student_id": "Synthetic identifier; it has no relationship to a real person.",
    "final_outcome": "Simulated outcome at the end of the observation window.",
    "passed": "TRUE for grades A, B, or C in this demonstration.",
    "course_completed": "FALSE only when the simulated final grade is W.",
    "cumulative_credits": "Running total of successfully earned credits.",
    "cumulative_gpa": "Simulated cumulative GPA calculated from non-withdrawal grades.",
    "credential_before_transfer": "Whether an award had already been generated before the transfer event.",
}

dictionary_rows = []
for table_name, df in tables.items():
    for col in df.columns:
        dictionary_rows.append({
            "table_name": table_name,
            "table_description": descriptions[table_name],
            "column_name": col,
            "pandas_type": str(df[col].dtype),
            "example_value": "" if df.empty else str(df[col].iloc[0]),
            "column_note": column_notes.get(col, ""),
        })

data_dictionary = pd.DataFrame(dictionary_rows)
data_dictionary.to_csv(project_dir / "data_dictionary.csv", index=False)


In [34]:

# Derived project metrics. These are descriptive properties of the synthetic simulation,
# not estimates of real student behavior.
students = tables["students"].copy()
enrollments = tables["enrollments"]
student_terms = tables["student_terms"]
service_usage = tables["service_usage"]
programs = tables["programs"]

passed_courses = (
    enrollments.loc[enrollments["passed"]]
    .groupby("student_id")["course_id"]
    .agg(set)
)

summary = students[[
    "student_id", "final_outcome", "initial_program_id",
    "first_generation", "pell_eligible", "full_time_at_entry"
]].copy()

summary["passed_english"] = summary["student_id"].map(
    lambda sid: "C002" in passed_courses.get(sid, set())
)
summary["passed_college_math"] = summary["student_id"].map(
    lambda sid: "C005" in passed_courses.get(sid, set())
)
summary["completed_gateway_pair"] = (
    summary["passed_english"] & summary["passed_college_math"]
)
summary["transferred"] = summary["final_outcome"].str.contains("Transferred")
summary["stopped_out"] = summary["final_outcome"].eq("Stopped Out")

service_flags = (
    service_usage.pivot_table(
        index="student_id",
        columns="service_id",
        values="service_use_id",
        aggfunc="count",
        fill_value=0,
    )
    if not service_usage.empty
    else pd.DataFrame(index=summary["student_id"])
)

summary = summary.merge(
    service_flags,
    left_on="student_id",
    right_index=True,
    how="left",
).fillna(0)

summary["used_advising"] = summary.get("S001", 0) > 0
summary["used_tutoring"] = summary.get("S002", 0) > 0
summary["used_transfer_center"] = summary.get("S003", 0) > 0

max_credits = student_terms.groupby("student_id")["cumulative_credits"].max()
summary["max_credits"] = summary["student_id"].map(max_credits).fillna(0)

metric_rows = []
for flag, label in [
    ("completed_gateway_pair", "Completed English Composition I and College Algebra"),
    ("used_advising", "Used academic advising"),
    ("used_tutoring", "Used tutoring"),
    ("used_transfer_center", "Used the transfer center"),
]:
    for value, group in summary.groupby(flag):
        metric_rows.append({
            "comparison": label,
            "group": "Yes" if bool(value) else "No",
            "students": len(group),
            "transfer_rate": round(group["transferred"].mean(), 4),
            "stopout_rate": round(group["stopped_out"].mean(), 4),
            "average_max_credits": round(group["max_credits"].mean(), 2),
        })

project_metrics = pd.DataFrame(metric_rows)
project_metrics.to_csv(project_dir / "project_metrics.csv", index=False)


In [45]:
# Google Cloud settings
PROJECT_ID = "pj-test1-499320"
DATASET_NAME = "student_journey"
BUCKET_NAME = "sql_graph"


# Fully qualified BigQuery dataset name
DATASET_ID = f"{PROJECT_ID}.{DATASET_NAME}"

# Create the BigQuery dataset
load_sql = f"""
-- Create the BigQuery dataset
CREATE SCHEMA IF NOT EXISTS `{DATASET_ID}`
OPTIONS (
  location = "US"
);

-- CSV files should be uploaded to:
-- gs://{BUCKET_NAME}/
"""

# Create one LOAD DATA statement for every generated table
for table_name in table_names:
    load_sql += f"""

LOAD DATA OVERWRITE `{DATASET_ID}.{table_name}`
FROM FILES (
  format = 'CSV',
  uris = ['gs://{BUCKET_NAME}/{table_name}.csv'],
  skip_leading_rows = 1,
  source_column_match = 'NAME'
  
);
"""

# Save the SQL script
(sql_dir / "01_load_csv_to_bigquery.sql").write_text(load_sql.strip(), encoding="utf-8")

print(f"SQL file created: {output_file}")
print(f"BigQuery dataset: {DATASET_ID}")


SQL file created: sql\01_load_csv_to_bigquery.sql
BigQuery dataset: pj-test1-499320.student_journey


2369

In [43]:
# Property graph DDL.
graph_sql = f"""-- Replace PROJECT_ID before running.
-- BigQuery Graph was a Preview feature when this project was prepared.
-- Keep the graph and all referenced tables in the same BigQuery location.

CREATE OR REPLACE PROPERTY GRAPH
  `{DATASET_ID}.StudentJourneyGraph`
NODE TABLES (
  `{DATASET_ID}.students` AS Students
    KEY (student_id)
    LABEL Student PROPERTIES ALL COLUMNS,

  `{DATASET_ID}.courses` AS Courses
    KEY (course_id)
    LABEL Course PROPERTIES ALL COLUMNS,

  `{DATASET_ID}.programs` AS Programs
    KEY (program_id)
    LABEL Program PROPERTIES ALL COLUMNS,

  `{DATASET_ID}.terms` AS Terms
    KEY (term_id)
    LABEL Term PROPERTIES ALL COLUMNS,

  `{DATASET_ID}.institutions` AS Institutions
    KEY (institution_id)
    LABEL Institution PROPERTIES ALL COLUMNS,

  `{DATASET_ID}.services` AS Services
    KEY (service_id)
    LABEL SupportService PROPERTIES ALL COLUMNS
)
EDGE TABLES (
  `{DATASET_ID}.enrollments` AS Enrollments
    KEY (enrollment_id)
    SOURCE KEY (student_id) REFERENCES Students (student_id)
    DESTINATION KEY (course_id) REFERENCES Courses (course_id)
    LABEL EnrolledIn PROPERTIES ALL COLUMNS,

  `{DATASET_ID}.student_terms` AS StudentTerms
    KEY (student_term_id)
    SOURCE KEY (student_id) REFERENCES Students (student_id)
    DESTINATION KEY (term_id) REFERENCES Terms (term_id)
    LABEL AttendedTerm PROPERTIES ALL COLUMNS,

  `{DATASET_ID}.program_history` AS ProgramHistory
    KEY (declaration_id)
    SOURCE KEY (student_id) REFERENCES Students (student_id)
    DESTINATION KEY (program_id) REFERENCES Programs (program_id)
    LABEL DeclaredProgram PROPERTIES ALL COLUMNS,

  `{DATASET_ID}.service_usage` AS ServiceUsage
    KEY (service_use_id)
    SOURCE KEY (student_id) REFERENCES Students (student_id)
    DESTINATION KEY (service_id) REFERENCES Services (service_id)
    LABEL UsedService PROPERTIES ALL COLUMNS,

  `{DATASET_ID}.awards` AS Awards
    KEY (award_id)
    SOURCE KEY (student_id) REFERENCES Students (student_id)
    DESTINATION KEY (program_id) REFERENCES Programs (program_id)
    LABEL EarnedAward PROPERTIES ALL COLUMNS,

  `{DATASET_ID}.transfers` AS Transfers
    KEY (transfer_id)
    SOURCE KEY (student_id) REFERENCES Students (student_id)
    DESTINATION KEY (institution_id) REFERENCES Institutions (institution_id)
    LABEL TransferredTo PROPERTIES ALL COLUMNS,

  `{DATASET_ID}.program_requirements` AS ProgramRequirements
    KEY (requirement_id)
    SOURCE KEY (program_id) REFERENCES Programs (program_id)
    DESTINATION KEY (course_id) REFERENCES Courses (course_id)
    LABEL Requires PROPERTIES ALL COLUMNS,

  `{DATASET_ID}.course_prerequisites` AS CoursePrerequisites
    KEY (prerequisite_id)
    SOURCE KEY (source_course_id) REFERENCES Courses (course_id)
    DESTINATION KEY (destination_course_id) REFERENCES Courses (course_id)
    LABEL Precedes PROPERTIES ALL COLUMNS
);
"""
(sql_dir / "02_create_property_graph.sql").write_text(graph_sql, encoding="utf-8")

3168

In [53]:

# Demonstration queries: GQL for relationships and GoogleSQL for validation.
query_sql = r"""-- Replace PROJECT_ID before running.

-- 1. Students who completed both gateway courses and later transferred.
GRAPH `pj-test1-499320.student_journey.StudentJourneyGraph`
MATCH
  (s:Student)-[english_enrollment:EnrolledIn]->
  (english:Course {course_id: "C002"}),
  (s)-[math_enrollment:EnrolledIn]->
  (math:Course {course_id: "C005"}),
  (s)-[transfer:TransferredTo]->
  (institution:Institution)
WHERE
  english_enrollment.passed = TRUE
  AND math_enrollment.passed = TRUE
RETURN
  s.student_id,
  institution.institution_name,
  transfer.transfer_term_id,
  transfer.credits_at_transfer,
  transfer.gpa_at_transfer
ORDER BY transfer.credits_at_transfer DESC;


-- 2. Courses most frequently completed by students who transferred.
GRAPH `pj-test1-499320.student_journey.StudentJourneyGraph`
MATCH
  (s:Student)-[enrollment:EnrolledIn]->(course:Course),
  (s)-[:TransferredTo]->(:Institution)
WHERE enrollment.passed = TRUE
RETURN
  course.course_id,
  course.course_name,
  COUNT(DISTINCT s.student_id) AS transferred_students
GROUP BY course.course_id, course.course_name
ORDER BY transferred_students DESC
LIMIT 15;


-- 3. Support services used by transferred students.
GRAPH `pj-test1-499320.student_journey.StudentJourneyGraph`
MATCH
  (s:Student)-[:UsedService]->(service:SupportService),
  (s)-[:TransferredTo]->(:Institution)
RETURN
  service.service_name,
  COUNT(DISTINCT s.student_id) AS transferred_students_using_service
GROUP BY service.service_name
ORDER BY transferred_students_using_service DESC;


-- 4. Required courses that a selected student completed.
-- Replace STU00001 with a student identifier returned by query 1.
GRAPH `pj-test1-499320.student_journey.StudentJourneyGraph`
MATCH
  (student:Student {student_id: "STU00001"})
    -[enrollment:EnrolledIn]->(course:Course)
    <-[:Requires]-(program:Program)
WHERE enrollment.passed = TRUE
RETURN
  student.student_id,
  program.program_name,
  course.course_id,
  course.course_name,
  enrollment.term_id,
  enrollment.final_grade
ORDER BY enrollment.term_id, course.course_id;


-- 5. Course-sequence paths of one to three prerequisite relationships.
GRAPH `pj-test1-499320.student_journey.StudentJourneyGraph`
MATCH
  (start:Course {course_id: "C004"})
  -[sequence:Precedes]->{1,3}
  (destination:Course)
RETURN
  start.course_name AS starting_course,
  destination.course_name AS reachable_course,
  ARRAY_LENGTH(sequence) AS sequence_length
ORDER BY sequence_length, reachable_course;


-- 6. Relational validation query.
-- This demonstrates why the graph should complement, not replace, ordinary SQL.
WITH student_flags AS (
  SELECT
    s.student_id,
    LOGICAL_OR(su.service_id = "S003") AS used_transfer_center,
    COUNTIF(t.transfer_id IS NOT NULL) > 0 AS transferred
  FROM `pj-test1-499320.student_journey.students` AS s
  LEFT JOIN `pj-test1-499320.student_journey.service_usage` AS su
    USING (student_id)
  LEFT JOIN `pj-test1-499320.student_journey.transfers` AS t
    USING (student_id)
  GROUP BY s.student_id
)
SELECT
  used_transfer_center,
  COUNT(*) AS students,
  AVG(CAST(transferred AS INT64)) AS descriptive_transfer_rate
FROM student_flags
GROUP BY used_transfer_center
ORDER BY used_transfer_center;
"""
(sql_dir / "03_example_graph_queries.sql").write_text(query_sql, encoding="utf-8")

3214

In [63]:
# Transfer profile for each institution: This compares transfer destinations based on student volume, credits, GPA, and credential completion before transfer.
query_sql = r"""-- Replace PROJECT_ID before running.
GRAPH `pj-test1-499320.student_journey.StudentJourneyGraph`

MATCH
  (student:Student)
    -[transfer:TransferredTo]->
  (institution:Institution)

RETURN
  institution.institution_name,
  institution.institution_group,
  COUNT(DISTINCT student.student_id) AS transferred_students,
  ROUND(AVG(transfer.credits_at_transfer), 1) AS average_credits,
  ROUND(AVG(transfer.gpa_at_transfer), 2) AS average_gpa,
  ROUND(
    100 * AVG(CAST(transfer.credential_before_transfer AS INT64)),
    1
  ) AS percent_credentialed_before_transfer

GROUP BY
  institution.institution_name,
  institution.institution_group

ORDER BY transferred_students DESC;
"""
(sql_dir / "04_ Transfer profile for each institution.sql").write_text(query_sql, encoding="utf-8")

681

In [3]:

## 1. Which programs feed the most students into which universities?
"""

**Question:** For every final program a student declared, which transfer institutions did those students end up at, and how many?

**Why this needs a graph:** This is really two hops chained together — student → program, and the same student → institution — grouped jointly. It's answerable in SQL with two joins, but the graph pattern states the relationship as directly as the question itself.
"""
sql_1=f"""
-- Replace PROJECT_ID before running.
GRAPH `pj-test1-499320.student_journey.StudentJourneyGraph`

MATCH
  -- Hop 1: find the student's FINAL declared program (not an earlier, changed one)
  (student:Student)
    -[declaration:DeclaredProgram]->
  (program:Program),

  -- Hop 2: find where that same student later transferred
  (student)
    -[transfer:TransferredTo]->
  (institution:Institution)

-- Only count the program a student ended on, so a student who
-- switched majors twice isn't credited to their abandoned program
WHERE declaration.declaration_status = "Final Program"

RETURN
  program.program_name,
  institution.institution_name,
  COUNT(DISTINCT student.student_id) AS transferred_students,   -- how many students made this exact trip
  ROUND(AVG(transfer.credits_at_transfer), 1) AS average_transfer_credits,
  ROUND(AVG(transfer.gpa_at_transfer), 2) AS average_transfer_gpa

GROUP BY
  program.program_name,
  institution.institution_name

ORDER BY transferred_students DESC
LIMIT 25;
"""
(sql_dir / "Q1. Which programs feed the most students into which universities.sql").write_text(sql_1, encoding="utf-8")



1008

In [9]:
## 2. What does each transfer institution's incoming class actually look like?
"""

**Question:** Across all students who transferred to a given institution, how many arrived, with how many credits, what GPA, and had they already earned a credential first?

**Why this needs a graph:** Only one hop is involved here (student → institution), so this is close to a table-shaped question — but it's still expressed as a graph query because it lives in the same graph and the aggregation logic (percent credentialed before transfer) reads cleanly off the edge property.
"""

sql_2=f"""-- Replace PROJECT_ID before running.
GRAPH `pj-test1-499320.student_journey.StudentJourneyGraph`

MATCH
  (student:Student)
    -[transfer:TransferredTo]->
  (institution:Institution)

RETURN
  institution.institution_name,
  institution.institution_group,                                 -- e.g. public 4-year, private, etc.
  COUNT(DISTINCT student.student_id) AS transferred_students,
  ROUND(AVG(transfer.credits_at_transfer), 1) AS average_credits,
  ROUND(AVG(transfer.gpa_at_transfer), 2) AS average_gpa,

  -- credential_before_transfer is a boolean edge property;
  -- casting to INT64 lets AVG() double as "percent TRUE"
  ROUND(
    100 * AVG(CAST(transfer.credential_before_transfer AS INT64)),
    1
  ) AS percent_credentialed_before_transfer

GROUP BY
  institution.institution_name,
  institution.institution_group

ORDER BY transferred_students DESC; 
"""
(sql_dir / "Q2. What does each transfer institution's incoming class actually look like.sql").write_text(sql_2, encoding="utf-8")

"""

**Reading the result:** An institution with high volume but a low `percent_credentialed_before_transfer` is receiving mostly students who left without finishing a credential first — a natural opening to discuss reverse-transfer or credential-completion partnerships.

"""


'\n\n**Reading the result:** An institution with high volume but a low `percent_credentialed_before_transfer` is receiving mostly students who left without finishing a credential first — a natural opening to discuss reverse-transfer or credential-completion partnerships.\n\n'

In [12]:
## 3. Where do students go when they change majors, and does it help them transfer?
"""
**Question:** Which program-to-program changes are most common, and of the students who make each specific change, how many go on to transfer?

**Why this needs a graph:** This requires matching the *same student* to two different declarations with two different statuses — one "Changed Program" and one "Final Program" — and comparing them. That's a same-node, two-edge pattern, which is exactly what graph matching is built for.

"""
sql_3=f"""-- Replace PROJECT_ID before running.
GRAPH `pj-test1-499320.student_journey.StudentJourneyGraph`

MATCH
  -- The program the student declared before changing
  (student:Student)
    -[old_declaration:DeclaredProgram]->
  (old_program:Program),

  -- The program the same student ended on
  (student)
    -[final_declaration:DeclaredProgram]->
  (final_program:Program)

WHERE
  old_declaration.declaration_status = "Changed Program"
  AND final_declaration.declaration_status = "Final Program"
  AND old_program.program_id <> final_program.program_id      -- exclude students who "changed" into the same program

RETURN
  old_program.program_name AS previous_program,
  final_program.program_name AS final_program,
  COUNT(DISTINCT student.student_id) AS students_changing_program,

  -- Count, within this same group, how many eventually transferred
  COUNT(
    DISTINCT IF(
      student.final_outcome IN (
        "Transferred",
        "Credentialed and Transferred"
      ),
      student.student_id,
      NULL
    )
  ) AS students_later_transferred

GROUP BY
  old_program.program_name,
  final_program.program_name

ORDER BY students_changing_program DESC;
"""
(sql_dir / "Q3. Where do students go when they change majors, and does it help them transfer.sql").write_text(sql_3, encoding="utf-8")

"""
```
**Reading the result:** Look for high-volume transitions (e.g., Engineering → Computer Science) and compare `students_later_transferred` against `students_changing_program` as an informal success rate for that specific pivot.

---
"""

'\n```\n**Reading the result:** Look for high-volume transitions (e.g., Engineering → Computer Science) and compare `students_later_transferred` against `students_changing_program` as an informal success rate for that specific pivot.\n\n---\n'

In [13]:
## 4. Which programs overlap enough in their curriculum to share students easily?
"""
**Question:** Which pairs of programs require many of the same courses?

**Why this needs a graph:** This is a *triangle* pattern — program 1 requires a course, and program 2 requires the *same* course — which means matching two different programs through a shared course node. Expressing "any two programs connected through any shared course" without knowing in advance how many shared courses exist is a natural fit for graph traversal, not a fixed join.
"""
sql_4=f"""-- Replace PROJECT_ID before running.
GRAPH `pj-test1-499320.student_journey.StudentJourneyGraph`

MATCH
  -- Program 1 requires a course...
  (program1:Program)
    -[:Requires]->
  (course:Course)
    -- ...and the SAME course is required by program 2 (edge direction reversed)
    <-[:Requires]-
  (program2:Program)

-- Avoid counting each pair twice (A,B and B,A) and avoid comparing a program to itself
WHERE program1.program_id < program2.program_id

RETURN
  program1.program_name AS first_program,
  program2.program_name AS second_program,
  COUNT(DISTINCT course.course_id) AS shared_required_courses,
  ARRAY_AGG(
    DISTINCT course.course_name
    ORDER BY course.course_name
  ) AS shared_courses                                          -- lists exactly which courses overlap

GROUP BY
  program1.program_name,
  program2.program_name

ORDER BY shared_required_courses DESC;
"""
(sql_dir / "Q4. Which programs overlap enough in their curriculum to share students easily.sql").write_text(sql_4, encoding="utf-8")

"""   
**Reading the result:** Program pairs near the top are candidates for stackable credentials, shared course scheduling, or an easier advising path when a student wants to switch between them.

---
"""

'   \n**Reading the result:** Program pairs near the top are candidates for stackable credentials, shared course scheduling, or an easier advising path when a student wants to switch between them.\n\n---\n'

In [14]:
## 5. Which pairs of courses show up together most often among students who transferred?
"""
**Question:** Among students who eventually transferred, which two courses did they most commonly both pass?

**Why this needs a graph:** This connects one student to two different courses *and* to a transfer event, all through the same student node — a three-way fan-out from one entity that would otherwise require self-joining the enrollment table twice.
"""
sql_5=f"""-- Replace PROJECT_ID before running.
GRAPH `pj-test1-499320.student_journey.StudentJourneyGraph`

MATCH
  -- The same student enrolled in course 1...
  (student:Student)
    -[enrollment1:EnrolledIn]->
  (course1:Course),

  -- ...and also enrolled in course 2...
  (student)
    -[enrollment2:EnrolledIn]->
  (course2:Course),

  -- ...and also transferred somewhere (destination not needed here, so left unnamed)
  (student)
    -[:TransferredTo]->
  (:Institution)

WHERE
  enrollment1.passed = TRUE
  AND enrollment2.passed = TRUE
  AND course1.course_id < course2.course_id                   -- avoid double-counting the pair in both orders

RETURN
  course1.course_name AS first_course,
  course2.course_name AS second_course,
  COUNT(DISTINCT student.student_id) AS transferred_students

GROUP BY
  course1.course_name,
  course2.course_name

ORDER BY transferred_students DESC
LIMIT 30;
"""
(sql_dir / "Q5. Which pairs of courses show up together most often among students who transferred.sql").write_text(sql_5, encoding="utf-8")
"""
**Reading the result:** A frequent course pair suggests a common academic path among transfer students — an association worth investigating, not proof that those two courses caused the transfer.

---
"""

'\n**Reading the result:** A frequent course pair suggests a common academic path among transfer students — an association worth investigating, not proof that those two courses caused the transfer.\n\n---\n'

In [ ]:

## 6. Which required courses are quietly derailing the most students?

**Question:** Of all courses that some program requires, which ones have the most students failing, withdrawing, or getting a D or F?

**Why this needs a graph:** This links a course to *every* program that requires it and to *every* student enrollment that failed, then aggregates across both — a many-to-many fan-in around a single course node.

```sql
GRAPH `pj-test1-499320.student_journey.StudentJourneyGraph`

MATCH
  -- A program requires this course...
  (program:Program)
    -[requirement:Requires]->
  (course:Course)
    -- ...and a student enrolled in it (edge direction reversed: student -> course)
    <-[enrollment:EnrolledIn]-
  (student:Student)

WHERE
  requirement.requirement_type = "Required"     -- exclude electives
  AND enrollment.passed = FALSE                  -- only look at unsuccessful attempts

RETURN
  course.course_id,
  course.course_name,
  course.course_group,
  COUNT(DISTINCT program.program_id) AS programs_requiring_course,  -- how "load-bearing" the course is
  COUNT(DISTINCT student.student_id) AS affected_students,
  COUNTIF(enrollment.final_grade = "W") AS withdrawals,
  COUNTIF(enrollment.final_grade = "D") AS grade_d,
  COUNTIF(enrollment.final_grade = "F") AS grade_f

GROUP BY
  course.course_id,
  course.course_name,
  course.course_group

ORDER BY affected_students DESC;
```

**Reading the result:** The most concerning rows combine a high `programs_requiring_course` (many students are funneled through it) with a high `affected_students` and a heavy tilt toward `withdrawals` — a strong candidate for a scheduling, prerequisite, or instructional review.

---


In [ ]:

## 7. How does support-service use differ by how a student's story ends?

**Question:** For each support service, how does usage differ between students who transferred, earned a credential, stopped out, or are still enrolled?

**Why this needs a graph:** One hop (student → service), but the interesting part is separating *how many students* used a service from *how many times* it was used — a distinction that's easy to blur in a simple table but stays clean when `usage` is treated as its own countable edge.

```sql
GRAPH `pj-test1-499320.student_journey.StudentJourneyGraph`

MATCH
  (student:Student)
    -[usage:UsedService]->
  (service:SupportService)

RETURN
  service.service_name,
  service.service_category,
  student.final_outcome,                                 -- Transferred, Credentialed, Stopped Out, Active, etc.
  COUNT(DISTINCT student.student_id) AS students,        -- unique people
  COUNT(usage) AS total_visits,                           -- every visit, including repeats by the same student
  ROUND(AVG(usage.minutes), 1) AS average_visit_minutes

GROUP BY
  service.service_name,
  service.service_category,
  student.final_outcome

ORDER BY
  service.service_name,
  students DESC;
```

**Reading the result:** If `total_visits` is much higher than `students` for one outcome group, that group is returning to the service repeatedly rather than just trying it once — a different story than a simple usage count would tell.

---


In [ ]:

## 8. Which combinations of support services show up together among transfer students?

**Question:** Do students who transfer tend to use certain services in pairs — advising plus tutoring, advising plus the transfer center?

**Why this needs a graph:** Same shape as query 5, but for services instead of courses — one student, two different service edges, plus a transfer edge, all fanning out from the same node.

```sql
GRAPH `pj-test1-499320.student_journey.StudentJourneyGraph`

MATCH
  (student:Student)
    -[:UsedService]->
  (service1:SupportService),

  (student)
    -[:UsedService]->
  (service2:SupportService),

  (student)
    -[:TransferredTo]->
  (:Institution)

WHERE service1.service_id < service2.service_id            -- avoid duplicate pairs and self-pairs

RETURN
  service1.service_name AS first_service,
  service2.service_name AS second_service,
  COUNT(DISTINCT student.student_id) AS transferred_students

GROUP BY
  service1.service_name,
  service2.service_name

ORDER BY transferred_students DESC;
```

**Reading the result:** A combination that appears far more often than either service alone suggests those two supports may function as a bundle in a successful path, worth testing as a recommended pairing in advising practice.

---


In [ ]:

## 9. What's the strongest program → course → institution "success chain"?

**Question:** For a given program, which specific completed course and which specific transfer destination show up together most often?

**Why this needs a graph:** Three different node types (program, course, institution) all connected through one student is a three-hop star pattern — the kind of question that turns into three separate joins in SQL but stays a single readable pattern here.

```sql
GRAPH `pj-test1-499320.student_journey.StudentJourneyGraph`

MATCH
  -- The student's final program...
  (student:Student)
    -[declaration:DeclaredProgram]->
  (program:Program),

  -- ...a course they passed while in that program...
  (student)
    -[enrollment:EnrolledIn]->
  (course:Course),

  -- ...and where they eventually transferred
  (student)
    -[:TransferredTo]->
  (institution:Institution)

WHERE
  declaration.declaration_status = "Final Program"
  AND enrollment.passed = TRUE
  AND enrollment.program_id = program.program_id      -- make sure the course counts toward THIS program, not a prior one

RETURN
  program.program_name,
  course.course_name,
  institution.institution_name,
  COUNT(DISTINCT student.student_id) AS transferred_students

GROUP BY
  program.program_name,
  course.course_name,
  institution.institution_name

ORDER BY transferred_students DESC
LIMIT 40;
```

**Reading the result:** A top row like `Computer Science → Programming Fundamentals II → Lone Star Technical University` describes a concrete, repeatable success pattern — useful for advising talking points or a targeted articulation agreement.

---


In [ ]:

## 10. What's the shortest curriculum path out of developmental math?

**Question:** Starting from Developmental Mathematics, what's the shortest prerequisite chain to reach each downstream course?

**Why this needs a graph:** This is a genuine variable-length path question — the number of steps between developmental math and, say, Calculus II isn't known in advance. `ANY SHORTEST` searches the prerequisite chain without anyone having to guess how many `Precedes` hops to write.

```sql
GRAPH `pj-test1-499320.student_journey.StudentJourneyGraph`

MATCH path = ANY SHORTEST
  (starting_course:Course {course_id: "C004"})    -- Developmental Mathematics
    -[:Precedes]->{1, 5}                           -- follow 1 to 5 Precedes hops, whichever is shortest
  (destination_course:Course)

RETURN
  starting_course.course_name AS starting_course,
  destination_course.course_name AS destination_course,
  PATH_LENGTH(path) AS number_of_steps

ORDER BY
  number_of_steps,
  destination_course;
```

**Reading the result:** A short result set like Developmental Math → College Algebra (1 step) → Calculus I (2 steps) → Calculus II (3 steps) gives advisors an at-a-glance curriculum map without hand-drawing it from the course catalog.

---


In [ ]:

## 11. Which students actually completed the whole programming sequence, in order?

**Question:** Which students passed Programming Fundamentals I, then Programming Fundamentals II, then Data Structures — and when?

**Why this needs a graph:** This combines the *formal* prerequisite chain (course1 → course2 → course3 via `Precedes`) with each individual student's *actual* enrollment history, checking that one specific student walked that exact path successfully.

```sql
GRAPH `pj-test1-499320.student_journey.StudentJourneyGraph`

MATCH
  -- Confirm the formal prerequisite chain exists...
  (student:Student)
    -[first_enrollment:EnrolledIn]->
  (course1:Course {course_id: "C019"})            -- Programming Fundamentals I
    -[:Precedes]->
  (course2:Course {course_id: "C020"})            -- Programming Fundamentals II
    -[:Precedes]->
  (course3:Course {course_id: "C021"}),           -- Data Structures

  -- ...and that the SAME student actually enrolled in course 2...
  (student)
    -[second_enrollment:EnrolledIn]->
  (course2),

  -- ...and course 3
  (student)
    -[third_enrollment:EnrolledIn]->
  (course3)

WHERE
  first_enrollment.passed = TRUE
  AND second_enrollment.passed = TRUE
  AND third_enrollment.passed = TRUE

RETURN
  student.student_id,
  student.final_outcome,
  first_enrollment.term_id AS programming_1_term,
  second_enrollment.term_id AS programming_2_term,
  third_enrollment.term_id AS data_structures_term

ORDER BY student.student_id;
```

**Reading the result:** The term columns let you see how many terms each student took to move through the sequence — a big gap between `programming_1_term` and `programming_2_term` might flag a student who paused or repeated a course along the way.

---


In [ ]:

## 12. What are the top five feeder courses into each transfer institution?

**Question:** For every transfer institution, which five completed courses show up most often among the students who transferred there?

**Why this needs a graph, plus a window function:** The graph pattern (student → course, student → institution) finds the raw counts, but *ranking within each institution* is a job for ordinary SQL. `GRAPH_TABLE` bridges the two: it runs the graph match and hands the result to regular BigQuery SQL as a normal table.

```sql
WITH course_feeders AS (
  SELECT *
  FROM GRAPH_TABLE(
    `pj-test1-499320.student_journey.StudentJourneyGraph`

    MATCH
      (student:Student)
        -[enrollment:EnrolledIn]->
      (course:Course),

      (student)
        -[:TransferredTo]->
      (institution:Institution)

    WHERE enrollment.passed = TRUE

    RETURN
      institution.institution_name AS institution_name,
      course.course_id AS course_id,
      course.course_name AS course_name,
      COUNT(DISTINCT student.student_id) AS transferred_students

    GROUP BY
      institution.institution_name,
      course.course_id,
      course.course_name
  )
)

SELECT
  institution_name,
  course_id,
  course_name,
  transferred_students
FROM course_feeders

-- QUALIFY + ROW_NUMBER() is plain SQL, applied AFTER the graph match is done —
-- this is the "regular SQL" half of the hybrid query
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY institution_name
  ORDER BY transferred_students DESC, course_name
) <= 5

ORDER BY
  institution_name,
  transferred_students DESC;
```

**Reading the result:** Exactly five rows per institution, ranked — a clean input for a dashboard table or a slide, without any manual top-N filtering after the fact.

---

## 13. Does using the transfer center actually line up with a higher transfer rate?

**Question:** Comparing students who used the transfer center against students who did not, what's the difference in transfer rate?

**Why this needs a graph, plus a left join:** A pure graph `MATCH` only returns students who *have* a `UsedService` edge — it can't produce students with *no* matching relationship. To include the "did not use" group, the graph result is extracted with `GRAPH_TABLE` and left-joined back to the full student table, so nonusers show up as `NULL` and get relabeled.

```sql
WITH transfer_center_users AS (
  SELECT DISTINCT student_id
  FROM GRAPH_TABLE(
    `pj-test1-499320.student_journey.StudentJourneyGraph`

    MATCH
      (student:Student)
        -[:UsedService]->
      (service:SupportService {service_id: "S003"})   -- S003 = Transfer Center

    RETURN student.student_id AS student_id
  )
)

SELECT
  -- Anyone missing from transfer_center_users never matched the graph pattern,
  -- which is exactly how we detect "did not use"
  IF(
    transfer_center_users.student_id IS NULL,
    "Did not use transfer center",
    "Used transfer center"
  ) AS transfer_center_group,

  COUNT(*) AS students,

  COUNTIF(
    students.final_outcome IN (
      "Transferred",
      "Credentialed and Transferred"
    )
  ) AS transferred_students,

  ROUND(
    100 * AVG(
      CAST(
        students.final_outcome IN (
          "Transferred",
          "Credentialed and Transferred"
        )
        AS INT64
      )
    ),
    1
  ) AS descriptive_transfer_rate

FROM `pj-test1-499320.student_journey.students` AS students

LEFT JOIN transfer_center_users
  USING (student_id)

GROUP BY transfer_center_group;
```

**Reading the result:** Two rows, one comparison — but remember the synthetic generator was deliberately built to make this association appear. Real institutional data would need the same caveat: this shows a designed (or observed) correlation, not proof that the transfer center *causes* transfer.

---

**One shared performance note:** queries 1, 3, 6, 9, and 13 each start from a specific, narrow anchor — a status filter, a specific course ID, a specific service ID — rather than matching every student first and filtering afterward. Google's own guidance recommends this pattern (start selective, prefer one `MATCH` over several) for exactly the reason these examples show: the earlier the graph engine can narrow the traversal, the less it has to enumerate before your `WHERE` clause even applies.

In [ ]:


article_md = """# The Student Journey Is Not a Spreadsheet

## Using BigQuery Graph to Connect the Dots in Higher Education

Higher-education institutions rarely suffer from a complete absence of data. They have student records, course registrations, grades, program declarations, advising visits, credentials, and transfer records. The harder problem is that these facts usually live in separate tables, and the question we want to answer is often about the path connecting them.

A dashboard can tell us how many students transferred. A graph-oriented analysis can help us investigate which course sequences, support interactions, program changes, and credit milestones commonly appeared before transfer.

This project demonstrates that idea with a fully synthetic community-college dataset. No record represents a real student, college, or university.

## Why use a graph?

SQL remains the correct tool for counts, rates, compliance reporting, and well-defined joins. A graph becomes useful when the analytical object is a relationship or path:

- a student completed one course before another;
- a program requires a connected set of courses;
- a student used several support services before reaching a milestone;
- many students followed similar routes into the same transfer institution.

The project therefore keeps ordinary BigQuery tables as the source of truth and defines a property graph over them. The graph is an analytical layer, not a replacement data warehouse.

## Model choices

The graph uses six node types: Student, Course, Program, Term, Institution, and SupportService.

The principal edge types are EnrolledIn, AttendedTerm, DeclaredProgram, UsedService, EarnedAward, TransferredTo, Requires, and Precedes.

This design intentionally avoids creating a node for every grade or visit. Those facts are properties of relationships. A grade describes a student's enrollment in a course; it is not an independent entity. A transfer term describes the transfer relationship; it is not the transfer destination itself.

## Synthetic-data logic

The generator creates 1,500 fictional students and simulates enrollment from Fall 2022 through Spring 2026. Each student receives latent preparation and engagement values that are never exported. Those hidden values influence course performance and persistence, while observable factors such as workload, enrollment intensity, advising, tutoring, gateway-course completion, accumulated credits, and transfer-center use shape later events.

The simulation intentionally contains associations that can be discovered by analysis. For example, students who complete English Composition I and College Algebra tend to persist longer. Students who reach the transfer center are more likely to have a transfer event. These are design choices made to produce a useful demonstration; they are not causal claims and should never be presented as findings about real students.

## Questions the project can explore

1. Which gateway-course combinations commonly appear before transfer?
2. Which support services are connected with longer enrollment paths?
3. Which programs send students to the widest range of institutions?
4. What prerequisite paths connect developmental education to advanced courses?
5. Which courses are shared across several successful program pathways?
6. Where do students commonly stop progressing through a required sequence?

## Responsible interpretation

A graph can make a path visually persuasive even when the underlying relationship is only descriptive. Sequence does not prove causation. Students who visit a transfer center may already be more transfer-oriented, and students who receive tutoring may differ from students who do not.

In a real institution, the next step would be to validate definitions, establish an observation window, control access, suppress small groups, de-identify student records, and involve institutional researchers, advisors, faculty, privacy officers, and students in interpreting results.

## Project contents

- `data/`: all synthetic CSV files
- `generate_synthetic_student_journey.py`: deterministic data generator
- `data_dictionary.csv`: table and column descriptions
- `project_metrics.csv`: descriptive checks from the simulation
- `sql/01_load_csv_to_bigquery.sql`: loading template
- `sql/02_create_property_graph.sql`: property-graph definition
- `sql/03_example_graph_queries.sql`: example GQL and SQL
"""

(project_dir / "article.md").write_text(article_md, encoding="utf-8")


4414

SyntaxError: incomplete input (2272003029.py, line 15)

In [ ]:

readme = f"""
# Synthetic Student Journey Graph Project 
This repository-style folder accompanies the article **“The Student Journey Is Not a Spreadsheet.”**
## Important notice

All students, institutions, outcomes, and relationships are synthetic. The data was generated with a fixed random seed (`20260702`) for reproducibility. It must not be used to make claims about real students or institutions.
## Dataset size

| Table | Rows |
|---|---:|

""" + "\n".join( f"| `{name}` | {len(df):,} |" for name, df in tables.items()) + f"""
## Final simulated outcomes

{students["final_outcome"].value_counts().rename_axis("outcome").to_frame("students").to_markdown()}

## Recommended workflow

1. Review `data_dictionary.csv`.
2. Run `generate_synthetic_student_journey.py` if you want to reproduce the CSV files.
3. Upload the files in `data/` to a Cloud Storage folder.
4. Replace `PROJECT_ID` and `BUCKET_NAME` in `sql/01_load_csv_to_bigquery.sql`.
5. Run `sql/02_create_property_graph.sql`.
6. Explore the project with `sql/03_example_graph_queries.sql`.
7. Validate every graph result with ordinary SQL before interpreting it.

## Why both GQL and SQL?

Graph Query Language is useful for paths and relationship patterns. GoogleSQL is still preferable for standard aggregations, official metrics, quality checks, and reporting. The project intentionally demonstrates both.
## Simulation caveat
Relationships such as advising use, gateway completion, accumulated credits, and transfer were deliberately made statistically associated. This creates an educational dataset with discoverable patterns. It does not establish cause and effect.
"""
(project_dir / "README.md").write_text(readme, encoding="utf-8")



ImportError: Missing optional dependency 'tabulate'.  Use pip or conda to install tabulate.